<a href="https://colab.research.google.com/github/irullah/comic-yolo/blob/main/Ruru_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🛠️ Langkah 1: Instalasi Library & Dependensi
Sel ini menginstal pustaka **Ultralytics** (mesin deteksi objek YOLOv8) dan **IOPaint** (mesin *deep learning inpainting* yang dioptimalkan untuk GPU).

> **Catatan Penting:** Jika setelah instalasi muncul tombol atau peringatan **"Restart runtime"**, klik tombol restart tersebut **1 kali**, lalu lanjutkan langsung ke Langkah 2 (tidak perlu menjalankan ulang sel ini).

### 📂 Langkah 2: Menghubungkan Google Drive
Menghubungkan Google Drive ke sesi Colab untuk memuat file model kustom YOLO (`best.pt`) yang tersimpan di Drive Anda.

### 🚀 Langkah 3: Eksekusi Batch Cleaning (YOLO + Anime-LaMa)
Alur kerja otomatis untuk membersihkan teks pada gambar komik/webtoon dari file ZIP:
* **Deteksi Teks:** YOLOv8 memindai balon kata menggunakan teknik *sliding window* dengan koordinat poligon presisi.
* **Model AI `anime-lama`:** Menggunakan varian model LaMa yang dilatih khusus untuk memulihkan tekstur 2D, *screentone*, dan garis komik (*lineart*).
* **Fitur HDStrategy.CROP:** IOPaint otomatis memotong (*crop*) hanya pada area sekitar balon kata, menghapus teksnya, lalu menempelkannya kembali. Menjaga gambar tetap tajam 100% dan mencegah GPU kehabisan VRAM (*Out of Memory*).
* **Otomasi File:** Mendeteksi file ZIP di sesi, mengekstrak, memproses seluruh halaman, lalu mengemas hasilnya ke file `C.zip` yang terunduh otomatis.


In [1]:
# 1. Instal ultralytics & paket pendukung inti (kompatibel penuh dengan Python terbaru)
%pip install -q --prefer-binary ultralytics loguru yacs piexif easydict omegaconf typer-config python-socketio diffusers transformers accelerate peft safetensors

# 2. Instal iopaint tanpa menurunkan versi Pillow bawaan Colab
%pip install -q --no-deps iopaint controlnet-aux

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =====================================================================
# SCRIPT ALL-IN-ONE: COMIC CLEANER AUTO-DOWNLOAD (TANPA BLOCK 4)
# =====================================================================

import os
import glob
import cv2
import torch
import numpy as np
import zipfile
import shutil
from ultralytics import YOLO
from iopaint.model_manager import ModelManager
from iopaint.schema import InpaintRequest, HDStrategy
from google.colab import files, drive

# --- 1. INSTALASI OTOMATIS DEPENDENSI AMAN PYTHON 3.13 ---
print("[INFO] Memeriksa dan menyiapkan library pendukung...")
get_ipython().system('pip install -q --prefer-binary ultralytics loguru yacs piexif easydict omegaconf typer-config python-socketio diffusers transformers accelerate peft safetensors')
get_ipython().system('pip install -q --no-deps iopaint controlnet-aux')

# Fungsi pembantu Unicode (mendukung nama file Jepang/Korea/Mandarin)
def read_image_utf8(path):
    img_array = np.fromfile(path, np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

def write_image_utf8(path, img):
    ext = os.path.splitext(path)[1]
    is_success, im_buf_arr = cv2.imencode(ext, img)
    if is_success:
        im_buf_arr.tofile(path)

# --- 2. PERSIAPAN MESIN AI (IOPaint Anime-LaMa & Custom YOLO) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n[1/4] Menyiapkan Mesin IOPaint (anime-lama) pada {device}...")
iopaint_model = ModelManager(name="anime-lama", device=device)

iopaint_config = InpaintRequest(
    hd_strategy=HDStrategy.CROP,
    hd_strategy_crop_trigger_size=800,
    hd_strategy_crop_margin=128
)

# Deteksi letak file bestv4.pt (Cek lokal Colab atau Google Drive)
model_path = "/content/bestv4.pt"
if not os.path.exists(model_path):
    if not os.path.exists("/content/drive/MyDrive"):
        print("🔄 Menghubungkan ke Google Drive untuk mencari bestv4.pt...")
        drive.mount("/content/drive")

    exact_path = "/content/drive/MyDrive/Colab Notebooks/Comic Cleaner /bestv4.pt"
    if os.path.exists(exact_path):
        model_path = exact_path
    else:
        found_pts = glob.glob("/content/drive/MyDrive/**/bestv4.pt", recursive=True)
        if found_pts:
            model_path = found_pts[0]
        else:
            raise FileNotFoundError(
                "❌ File bestv4.pt tidak ditemukan! Silakan upload bestv4.pt "
                "ke panel Files Colab di sebelah kiri atau simpan di Google Drive."
            )

print(f"[2/4] Menyalakan Radar Deteksi Teks menggunakan: {model_path}")
text_detector = YOLO(model_path)

# --- 3. PILIH FILE ZIP KOMIK DI SESSION COLAB ---
input_dir = "input_komik"
output_dir = "output_komik"

zip_candidates = [
    f for f in os.listdir('.')
    if f.lower().endswith('.zip') and not f.startswith('C')
]

if not zip_candidates:
    raise FileNotFoundError(
        "❌ ERROR: Tidak ditemukan file .zip komik di panel Files Colab!\n"
        "💡 Silakan upload file .zip komik ke panel Files di sebelah kiri, lalu jalankan ulang cell ini."
    )

zip_path = zip_candidates[0]
if len(zip_candidates) > 1:
    print(f"\n[3/4] Ditemukan beberapa ZIP, menggunakan file pertama: {zip_path}")
else:
    print(f"\n[3/4] Menggunakan file ZIP terdeteksi: {zip_path}")

# Bersihkan direktori lama
if os.path.exists(input_dir): shutil.rmtree(input_dir)
if os.path.exists(output_dir): shutil.rmtree(output_dir)
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print(f"Mengekstrak {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(input_dir)

valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
image_files = []
for root, dirs, files_in_dir in os.walk(input_dir):
    for file in files_in_dir:
        if file.lower().endswith(valid_extensions):
            image_files.append(os.path.join(root, file))

image_files.sort()
total_images = len(image_files)

if total_images == 0:
    raise ValueError("❌ ERROR: Tidak ditemukan file gambar (.jpg/.jpeg/.png/.webp) di dalam file ZIP!")

# --- 4. PROSES PEMBERSIHAN TEKS (BATCH PROCESSING) ---
print(f"\n[4/4] Memulai Pembersihan Otomatis untuk {total_images} gambar...")

for idx, img_path in enumerate(image_files, 1):
    rel_path = os.path.relpath(img_path, input_dir)
    save_path = os.path.join(output_dir, rel_path)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    print(f"---> Memproses [{idx}/{total_images}]: {rel_path}...")

    img = read_image_utf8(img_path)
    if img is None:
        print(f"⚠️ Gagal membaca {rel_path}, dilewati.")
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask_teks = np.zeros(img.shape[:2], dtype="uint8")

    # Deteksi Sliding Window YOLO
    H, W = img_rgb.shape[:2]
    slice_h = W
    overlap = int(slice_h * 0.1)
    step = slice_h - overlap

    for y in range(0, H, step):
        y1 = y
        y2 = min(y + slice_h, H)
        if y2 - y1 < 50:
            break

        crop_img = img_rgb[y1:y2, 0:W]
        results = text_detector.predict(crop_img, conf=0.10, verbose=False)[0]

        if results.masks is not None:
            polygons = results.masks.xy
            for poly in polygons:
                poly_pts = np.array(poly, dtype=np.int32)
                poly_pts[:, 1] += y1
                cv2.fillPoly(mask_teks, [poly_pts], 255)

    # Jika tidak ada teks, langsung simpan gambar asli
    if not np.any(mask_teks > 0):
        write_image_utf8(save_path, img)
        print(f"⏩ {rel_path} tidak ada teks, langsung disimpan!")
        continue

    # Dilation Dinamis
    k_size = max(3, int(img.shape[1] * 0.010))
    if k_size % 2 == 0:
        k_size += 1
    kernel = np.ones((k_size, k_size), np.uint8)
    mask_final = cv2.dilate(mask_teks, kernel, iterations=1)

    # Inpainting dengan IOPaint Anime-LaMa
    hasil_bgr = iopaint_model(img_rgb, mask_final, iopaint_config)

    write_image_utf8(save_path, hasil_bgr)
    print(f"✅ {rel_path} selesai dibersihkan!")

# --- 5. BUNGKUS KE ZIP & DOWNLOAD OTOMATIS ---
print("\n📦 Membungkus hasil bersih ke file C.zip...")
output_zip_name = "C"
if os.path.exists(f"{output_zip_name}.zip"):
    os.remove(f"{output_zip_name}.zip")

shutil.make_archive(output_zip_name, 'zip', output_dir)

print("🚀 Memicu download otomatis ke perangkat Anda...")
files.download(f"{output_zip_name}.zip")

print("\n🎉 === SEMUA SELESAI! ===")